# Notebook 1 — Load Bronze Layer
**BankGuard | Microsoft Fabric**

## What does this notebook do?
This is the very first step. We take raw CSV files that have been uploaded to our Lakehouse and save them as **Delta tables**.

We do **no cleaning** here. We want to keep a copy of the data exactly as it came from the source system. This is the **Bronze layer** principle — preserve raw data.

### Files we expect (upload these to your Lakehouse Files section first):
- `Files/raw/transactions.csv`
- `Files/raw/customers.csv`
- `Files/raw/loans.csv`

## Step 1: Setup
Microsoft Fabric automatically creates a Spark session for you — you don't need to set it up manually.
The `spark` variable is already available.

In [ ]:
# These are standard Python imports
from pyspark.sql import functions as F   # F gives us SQL-like functions
import datetime

# Record when we ran this notebook — useful for debugging later
LOAD_DATE = datetime.date.today().strftime('%Y-%m-%d')

print(f'Load date: {LOAD_DATE}')
print('Spark session ready:', spark.version)

## Step 2: Load Transactions CSV

In Fabric, when your Lakehouse is attached to the notebook, you can read files using the `Files/` path.

`inferSchema=True` means Spark will try to guess the data types automatically.

In [ ]:
# Read the CSV file
# In Fabric: 'Files/raw/transactions.csv' refers to the Files section of your Lakehouse
df_transactions = (
    spark.read
         .option('header', True)          # First row is column names
         .option('inferSchema', True)     # Guess data types automatically
         .csv('Files/raw/transactions.csv')
)

# Always check how many rows you loaded
print(f'Rows loaded: {df_transactions.count():,}')

# Check the column names and types
df_transactions.printSchema()

In [ ]:
# Preview the first 5 rows
df_transactions.show(5, truncate=False)

In [ ]:
# Add metadata columns so we know when and where this data came from
# This is a best practice — always track where your data came from
df_transactions_bronze = df_transactions.withColumns({
    '_load_date'    : F.lit(LOAD_DATE),       # When was this loaded?
    '_source_file'  : F.lit('transactions.csv'),  # Which file did it come from?
})

# Save as a Delta table in the Lakehouse
# 'Tables/bronze_transactions' creates a table called bronze_transactions
(
    df_transactions_bronze
    .write
    .format('delta')       # Delta is Fabric's default format — gives us ACID transactions
    .mode('overwrite')     # Overwrite if table already exists (safe for full reload)
    .saveAsTable('bronze_transactions')   # Saves to the attached Lakehouse
)

print('✅ bronze_transactions saved!')

## Step 3: Load Customers CSV

In [ ]:
df_customers = (
    spark.read
         .option('header', True)
         .option('inferSchema', True)
         .csv('Files/raw/customers.csv')
)

print(f'Customers loaded: {df_customers.count():,}')
df_customers.printSchema()

In [ ]:
df_customers_bronze = df_customers.withColumns({
    '_load_date'    : F.lit(LOAD_DATE),
    '_source_file'  : F.lit('customers.csv'),
})

(
    df_customers_bronze
    .write.format('delta')
    .mode('overwrite')
    .saveAsTable('bronze_customers')
)

print('✅ bronze_customers saved!')

## Step 4: Load Loans CSV

In [ ]:
df_loans = (
    spark.read
         .option('header', True)
         .option('inferSchema', True)
         .csv('Files/raw/loans.csv')
)

print(f'Loans loaded: {df_loans.count():,}')
df_loans.printSchema()

In [ ]:
df_loans_bronze = df_loans.withColumns({
    '_load_date'    : F.lit(LOAD_DATE),
    '_source_file'  : F.lit('loans.csv'),
})

(
    df_loans_bronze
    .write.format('delta')
    .mode('overwrite')
    .saveAsTable('bronze_loans')
)

print('✅ bronze_loans saved!')

## Step 5: Quick Check — What Did We Create?

Let's confirm our 3 Bronze tables exist and have data.

In [ ]:
# List all tables in the current Lakehouse
print('Tables created in this Lakehouse:')
spark.sql('SHOW TABLES').show()

# Quick row count summary
tables = ['bronze_transactions', 'bronze_customers', 'bronze_loans']
for t in tables:
    count = spark.sql(f'SELECT COUNT(*) as cnt FROM {t}').collect()[0]['cnt']
    print(f'  {t}: {count:,} rows')

print('\n✅ Bronze layer complete! Run Notebook 2 next.')